### **Task 2:  Sentiment Analysis using NLP Pipeline  &  ML Models**

**PROJECT STRUCTURE**
1. Import Libraries
2. Load Dataset
3. Data Understanding
4. NLP Preprocessing
5. Feature Engineering (BoW + TF-IDF)
6. Model Building (3 Models)
7. Model Evaluation
8. Comparison & Insights
9. Final Conclusion

In [9]:
# 1.IMPORT LIBRARIES

import pandas as pd
import numpy as np
import re
import string

# NLP
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer

# Sklearn
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier

nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\prajy\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\prajy\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [10]:
# 2.LOAD DATASET

df = pd.read_csv("IMDB Dataset.csv")
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [11]:
# 3.DATA UNDERSTANDING

print(df.shape)
print(df['sentiment'].value_counts())

df.sample(5)

(50000, 2)
sentiment
positive    25000
negative    25000
Name: count, dtype: int64


,review,sentiment
685,The IMDb plot summary in no way describes the ...,negative
635,As the number of Video Nasties I've yet to see...,negative
3516,Alistair Simms inspired portrayal of Miss Frit...,positive
13933,"Directed by Michael Curtiz, Four Daughters is ...",positive
20721,"Not only is this film entertaining, with excel...",positive


In [12]:
# 4.NLP PREPROCESSING

stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    text = text.lower()
    
    text = re.sub(r"http\S+|www\S+", '', text)  # remove URLs
    text = re.sub(r'\d+', '', text)  # remove numbers
    text = text.translate(str.maketrans('', '', string.punctuation))
    
    words = text.split()
    
    words = [word for word in words if word not in stop_words]
    
    words = [lemmatizer.lemmatize(word) for word in words]
    
    return " ".join(words)

In [13]:
df['clean_text'] = df['review'].apply(preprocess_text)

In [15]:
# 5.FEATURE ENGINEERING
#Bag of Words
bow = CountVectorizer(max_features=5000)

X_bow = bow.fit_transform(df['clean_text'])

In [16]:
#TF-IDF
tfidf = TfidfVectorizer(max_features=5000)

X_tfidf = tfidf.fit_transform(df['clean_text'])

In [17]:
# 6.TRAIN TEST SPLIT

y = df['sentiment']

X_train_bow, X_test_bow, y_train, y_test = train_test_split(X_bow, y, test_size=0.2, random_state=42)
X_train_tfidf, X_test_tfidf, _, _ = train_test_split(X_tfidf, y, test_size=0.2, random_state=42)

In [18]:
# 7.MODEL BUILDING

#Logistic Regression
lr = LogisticRegression()
lr.fit(X_train_tfidf, y_train)

y_pred_lr = lr.predict(X_test_tfidf)

In [19]:
#Naive Bayes
nb = MultinomialNB()
nb.fit(X_train_bow, y_train)

y_pred_nb = nb.predict(X_test_bow)

In [20]:
#Decision Tree
dt = DecisionTreeClassifier()
dt.fit(X_train_tfidf, y_train)

y_pred_dt = dt.predict(X_test_tfidf)

In [22]:
# 8.MODEL EVALUATION

print("Logistic Regression")
print(accuracy_score(y_test, y_pred_lr))
print(classification_report(y_test, y_pred_lr))

print("Naive Bayes")
print(accuracy_score(y_test, y_pred_nb))
print(classification_report(y_test, y_pred_nb))

print("Decision Tree")
print(accuracy_score(y_test, y_pred_dt))
print(classification_report(y_test, y_pred_dt))

Logistic Regression
0.8846
              precision    recall  f1-score   support

    negative       0.89      0.87      0.88      4961
    positive       0.88      0.90      0.89      5039

    accuracy                           0.88     10000
   macro avg       0.88      0.88      0.88     10000
weighted avg       0.88      0.88      0.88     10000

Naive Bayes
0.8465
              precision    recall  f1-score   support

    negative       0.84      0.85      0.85      4961
    positive       0.85      0.84      0.85      5039

    accuracy                           0.85     10000
   macro avg       0.85      0.85      0.85     10000
weighted avg       0.85      0.85      0.85     10000

Decision Tree
0.7168
              precision    recall  f1-score   support

    negative       0.71      0.72      0.72      4961
    positive       0.72      0.71      0.72      5039

    accuracy                           0.72     10000
   macro avg       0.72      0.72      0.72     10000
weighte

In [23]:
# 9.COMPARISON TABLE

results = pd.DataFrame({
    'Model': ['Logistic Regression', 'Naive Bayes', 'Decision Tree'],
    'Accuracy': [
        accuracy_score(y_test, y_pred_lr),
        accuracy_score(y_test, y_pred_nb),
        accuracy_score(y_test, y_pred_dt)
    ]
})

results.sort_values(by='Accuracy', ascending=False)

,Model,Accuracy
0,Logistic Regression,0.8846
1,Naive Bayes,0.8465
2,Decision Tree,0.7168
